In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "midlands_subset.csv"

# Load the latest version
df_midlands = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "johntandy/epc-midlands-july-2023",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)


#1. Explore the data ETL
#Display field, check for missing rows,
print(df_midlands.dtypes)
df_midlands.describe(include='all')

# Display the first few rows to verify
df_midlands.head()

In [ ]:
## Section 1 - Exploring features that relate to energy.
## We're looking only at houses and the ones given a lower rating E to G.   
##  We'll see what causes the low rating very quick by the Percentage_Poor_or_Very_Poor scores.


# 1. Filter for Houses with low EPC ratings (E, F, or G)
df_houses = df_midlands[
    (df_midlands['PROPERTY_TYPE'] == 'House') &
    (df_midlands['CURRENT_ENERGY_RATING'].isin(['E', 'F', 'G']))
].copy()

# 2. Glazing setup
modern_glazing_types = [
    'double glazing installed during or after 2002',
    'triple glazing',
    'triple, known data'
]

# 3. Wall insulation setup
wall_u_values = df_houses['WALLS_DESCRIPTION'].str.extract(r'(\d+\.\d+)\s*W/m')[0].astype(float)

good_wall_terms = [
    'filled cavity',
    'insulated',
    'with external insulation',
    'with internal insulation',
    'with additional insulation'
]

# 4. Create Boolean Flags
df_houses['HAS_LOFT_300MM'] = df_houses['ROOF_DESCRIPTION'].str.contains(r'300|400|350|250', case=False, na=False)

df_houses['HAS_CAVITY_INS'] = (
    df_houses['WALLS_DESCRIPTION'].str.contains('|'.join(good_wall_terms), case=False, na=False) |
    (wall_u_values <= 0.35)
)

df_houses['HAS_MODERN_GLAZING'] = df_houses['GLAZED_TYPE'].isin(modern_glazing_types)

# 5. Calculate potential SAP point gain
df_houses['POTENTIAL_GAIN'] = df_houses['POTENTIAL_ENERGY_EFFICIENCY'] - df_houses['CURRENT_ENERGY_EFFICIENCY']

# 6. Filter unimproved houses (missing Loft AND missing Wall AND missing Glazing)
unimproved_filter = (
    (~df_houses['HAS_LOFT_300MM']) &
    (~df_houses['HAS_CAVITY_INS']) &
    (~df_houses['HAS_MODERN_GLAZING'])
)

unimproved = df_houses[unimproved_filter]

print(f"Average Potential SAP Point Gain: {unimproved['POTENTIAL_GAIN'].mean():.1f} points")

In [ ]:
def map_wall_score(text):
    if not isinstance(text, str):
        return 1
    text = text.lower()
    if "cavity" in text:
        return 3
    if "timber" in text:
        return 2
    if any(word in text for word in ["sandstone", "stone", "solid", "granite", "rubble"]):
        return 1
    return 1  # default to solid wall if unknown

# Add the new field "wall_score" to the unimproved DataFrame
unimproved = unimproved.copy()
unimproved["wall_score"] = unimproved["WALLS_DESCRIPTION"].apply(map_wall_score)

# Display a preview to verify the new column
unimproved[["WALLS_DESCRIPTION", "wall_score"]].head()

def roof_ins_score(text):
    if not isinstance(text, str):
        return 0
    t = text.lower()
    if "flat" in t:
        return 1
    if "pitched" in t:
        return 3
    if "roof room" in t:
        return 3
    if "thatched" in t:
        return 3
    return 0

unimproved = unimproved.copy()
unimproved["roof_score"] = unimproved["ROOF_DESCRIPTION"].apply(roof_ins_score)

def window_score(text):
    # Handle missing or non-string values
    if not isinstance(text, str):
        return 0

    t = text.lower()

    # --- SINGLE GLAZING (score = 3) ---
    if "single" in t:
        return 3

    # --- DOUBLE GLAZING (score = 2) ---
    if "double" in t:
        return 2

    # --- TRIPLE GLAZING (score = 0) ---
    if "triple" in t:
        return 0

    # --- UNKNOWN / OTHER (fallback) ---
    return 1

unimproved = unimproved.copy()
unimproved["window_score"] = unimproved["WINDOWS_DESCRIPTION"].apply(window_score)

In [ ]:
age_band_map = {
    # Numeric / Specific year codes
    '1820': '<1900',
    '1900': '1900-1929',
    '1935': '1930-1949',
    '1950': '1950-1966',
    '1960': '1950-1966',
    '1988': '1983-1990',
    '2010': '2009-2011',
    '2011': '2012-2023',
    '2012': '2012-2023',
    '2013': '2012-2023',
    '2014': '2012-2023',
    '2015': '2012-2023',
    '2016': '2012-2023',
    '2017': '2012-2023',
    '2018': '2012-2023',
    '2019': '2012-2023',
    '2020': '2012-2023',
    '2021': '2012-2023',
    '2022': '2012-2023',
    '2023': '2012-2023',

    # England and Wales textual descriptions
    'England and Wales: before 1900': '<1900',
    'England and Wales: 1900-1929': '1900-1929',
    'England and Wales: 1930-1949': '1930-1949',
    'England and Wales: 1950-1966': '1950-1966',
    'England and Wales: 1967-1975': '1967-1975',
    'England and Wales: 1976-1982': '1976-1982',
    'England and Wales: 1983-1990': '1983-1990',
    'England and Wales: 1991-1995': '1991-1995',
    'England and Wales: 1996-2002': '1996-2002',
    'England and Wales: 2003-2006': '2003-2006',
    'England and Wales: 2007 onwards': '2007-2009',
    'England and Wales: 2007-2011': '2009-2011',
    'England and Wales: 2012 onwards': '2012-2023',

    # Invalid / Missing entries
    'INVALID!': 'ignore',
    'NO DATA!': 'ignore'
}


unimproved = unimproved.copy()
unimproved['CONSTRUCTION_AGE_GROUP'] = unimproved['CONSTRUCTION_AGE_BAND'].astype(str).map(age_band_map).fillna('ignore')


In [ ]:
!pip install pgeocode
postcodes = unimproved["POSTCODE"].dropna().str.strip().str.upper().unique()
import pgeocode
import pandas as pd

nomi = pgeocode.Nominatim('gb')

lookup_rows = []

for pc in postcodes:
    result = nomi.query_postal_code(pc)
    lookup_rows.append({
        "POSTCODE": pc,
        "LAT": result.latitude,
        "LON": result.longitude
    })

postcode_lookup = pd.DataFrame(lookup_rows)


In [ ]:
# Calculate the combined retrofit cost score
unimproved["retrofit_cost_score"] = (
    unimproved["wall_score"] +
    unimproved["roof_score"] +
    unimproved["window_score"]
)

# Calculate the Easy Win Index
unimproved = unimproved.copy()
unimproved["easy_win_index"] = (
    unimproved["retrofit_cost_score"] * unimproved["POTENTIAL_GAIN"]
)




Use the newly calculated retrofit cost score which takes into account the cost of the improvement as well as the potential gain. We want to map those results externally to Python using the Lat and Lon values look up found from the postcode locations.

In [ ]:
import os

# Ensure the dataset directory exists
os.makedirs('/content/dataset', exist_ok=True)

# Define output path
output_path = '/content/dataset/unimproved_subset.csv'

# Save the dataframe
unimproved.to_csv(output_path, index=False)
print(f"DataFrame saved successfully to: {output_path}")

Potential improvement against postcode town.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Group the unimproved properties by POSTTOWN and aggregate including easy_win_index
town_summary = (
    unimproved.groupby('POSTTOWN')
    .agg(
        Average_Gain=('POTENTIAL_GAIN', 'mean'),
        Average_Easy_Win=('easy_win_index', 'mean'),
        House_Count=('POTENTIAL_GAIN', 'count'),
        Cost_Easy_Win=('retrofit_cost_score', 'mean'),  
    )
    .reset_index()
)

# 2. Filter for towns with a reasonable sample size (e.g., more than 50 houses)
min_houses_threshold = 50
town_summary_filtered = town_summary[town_summary['House_Count'] >= min_houses_threshold].sort_values(by='Average_Gain', ascending=False)

# 3. Create the figure and first axis (for bars)
fig, ax1 = plt.subplots(figsize=(15, 7))
sns.set_theme(style="whitegrid")

# Create bar plot on ax1
barplot = sns.barplot(
    data=town_summary_filtered,
    x='POSTTOWN',
    y='Average_Gain',
    palette='viridis',
    hue='Cost_Easy_Win',
    legend=False,
    ax=ax1
)

# Add exact value annotations on top of each bar
for p in barplot.patches:
    height = p.get_height()
    if height > 0:
        barplot.annotate(
            f"{height:.1f}",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom', fontsize=9, fontweight='bold', xytext=(0, 3), textcoords='offset points'
        )

# Create secondary axis for easy_win_index
ax2 = ax1.twinx()

# Create line plot on ax2
# We use the categorical x positions matching the bars
x_coords = range(len(town_summary_filtered))
sns.lineplot(
    x=x_coords,
    y=town_summary_filtered['Average_Easy_Win'],
    color='crimson',
    marker='o',
    linewidth=2.5,
    ax=ax2,
    label='Average Easy Win Index'
)

# Customize secondary y-axis
ax2.set_ylabel('Average Easy Win Index', color='crimson', fontsize=12, fontweight='bold', labelpad=10)
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.grid(False)  # Avoid overlapping grid lines

# Set labels and title
ax1.set_title('Average Potential Gain vs. Easy Win Index by Post Town (N >= 50)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Post Town (POSTTOWN)', fontsize=12, fontweight='bold', labelpad=10)
ax1.set_ylabel('Average Potential Gain (SAP Points)', fontsize=12, fontweight='bold')
ax1.set_xticks(x_coords)
ax1.set_xticklabels(town_summary_filtered['POSTTOWN'], rotation=45, ha='right')

# Adjust legend for the line plot
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# @title Select Construction Age Band
AgeBand = '1900-1929'  # @param ['All', '<1900', '1900-1929', '1930-1949', '1950-1966', '1967-1975', '1976-1982', '1983-1990', '1991-1995', '1996-2002', '2003-2006', '2012-2023']

# Filter the unimproved DataFrame based on the selected AgeBand
if AgeBand == 'All':
    unimproved_filtred = unimproved.copy()
else:
    unimproved_filtred = unimproved[unimproved['CONSTRUCTION_AGE_GROUP'] == AgeBand].copy()

# Display the size of the filtered DataFrame and a preview
print(f"Filtered properties for age group '{AgeBand}': {len(unimproved_filtred)}")
display(unimproved_filtred.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns



# 1. Group the unimproved properties by POSTTOWN and aggregate including easy_win_index and retrofit_cost_score
town_summary = (
    unimproved_filtred.groupby('POSTTOWN')
    .agg(
        Average_Gain=('POTENTIAL_GAIN', 'mean'),
        Average_Easy_Win=('easy_win_index', 'mean'),
        House_Count=('column00', 'count'),
        Cost_Easy_Win=('retrofit_cost_score', 'mean'),
    )
    .reset_index()
)

# 2. Filter for towns with a reasonable sample size (e.g., more than 50 houses)
min_houses_threshold = 50
town_summary_filtered = town_summary[town_summary['House_Count'] >= min_houses_threshold].sort_values(by='Average_Gain', ascending=False)

# 3. Create the figure and first axis (for bars)
fig, ax1 = plt.subplots(figsize=(16, 8))
sns.set_theme(style="whitegrid")

# Create bar plot on ax1
barplot = sns.barplot(
    data=town_summary_filtered,
    x='POSTTOWN',
    y='Average_Gain',
    palette='viridis',
    hue='Cost_Easy_Win',
    legend=True,
    ax=ax1
)

# Add exact value annotations on top of each bar
for p in barplot.patches:
    height = p.get_height()
    if height > 0:
        barplot.annotate(
            f"{height:.1f}",
            (p.get_x() + p.get_width() / 2., height),
            ha='center', va='bottom', fontsize=9, fontweight='bold', xytext=(0, 4), textcoords='offset points'
        )

# Create secondary axis for easy_win_index
ax2 = ax1.twinx()

# Create line plot on ax2 using categorical positions matching the bars
x_coords = range(len(town_summary_filtered))
sns.lineplot(
    x=x_coords,
    y=town_summary_filtered['Average_Easy_Win'],
    color='crimson',
    marker='o',
    linewidth=2.5,
    ax=ax2,
    label='Average Easy Win Index'
)

# Customize secondary y-axis
ax2.set_ylabel('Average Easy Win Index', color='crimson', fontsize=12, fontweight='bold', labelpad=10)
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.grid(False)  

# Set labels and title
ax1.set_title('Average Potential Gain vs. Easy Win Index by Post Town (N >= 50)', fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Post Town (POSTTOWN)', fontsize=12, fontweight='bold', labelpad=10)
ax1.set_ylabel('Average Potential Gain (SAP Points)', fontsize=12, fontweight='bold')
ax1.set_xticks(x_coords)
ax1.set_xticklabels(town_summary_filtered['POSTTOWN'], rotation=45, ha='right')

# Position the legends outside the main plot area to prevent overlap
ax1.legend(title='Avg Retrofit Cost Score', bbox_to_anchor=(1.12, 1.0), loc='upper left', borderaxespad=0.)
ax2.legend(bbox_to_anchor=(1.12, 0.5), loc='upper left', borderaxespad=0.)

plt.tight_layout()
plt.show()